<a href="https://colab.research.google.com/github/Vinz602/ResNet-50_Implementation/blob/main/ResNet_50_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import torch
import torch.nn as nn


In [7]:
class Bottleneck(nn.Module):
  #1x1 -> 3x3 -> 1x1 -> skip connection
  expansion = 4

  def __init__(self, in_channels, mid_channels, stride=1):
    super().__init__()
    out_channels = mid_channels * self.expansion

    #1x1
    self.conv1 = nn.Conv2d(in_channels, mid_channels, kernel_size=1, bias=False)
    self.bn1 = nn.BatchNorm2d(mid_channels)

    #3x3
    self.conv2 = nn.Conv2d(mid_channels, mid_channels, kernel_size=3, stride=stride, padding=1, bias=False)
    self.bn2 = nn.BatchNorm2d(mid_channels)

    #1x1
    self.conv3 = nn.Conv2d(mid_channels, out_channels, kernel_size=1, bias=False)
    self.bn3 = nn.BatchNorm2d(out_channels)
    self.relu = nn.ReLU(inplace=True)

    #Projection shortcut: when shape of x not the same as F(x)
    self.downsample = None
    if stride != 1 or in_channels != out_channels:
      self.downsample = nn.Sequential(
          nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
          nn.BatchNorm2d(out_channels)
      )

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    identity = x if self.downsample is None else self.downsample(x)

    out = self.relu(self.bn1(self.conv1(x)))
    out = self.relu(self.bn2(self.conv2(out)))
    out = self.bn3(self.conv3(out))

    out += identity
    out = self.relu(out)

    return out

In [8]:
class ResNet(nn.Module):
  def __init__(self, layers=(3,4,6,3), num_classes=1000, zero_init_residual=True):
    super().__init__()
    self.in_channels = 64

    #[B,3,224,224] -> [B,64,112,112] -> [B,64,56,56]
    self.conv1 = nn.Conv2d(3,64, kernel_size=7, stride=2, padding=3, bias=False)
    self.bn1 = nn.BatchNorm2d(64)
    self.relu = nn.ReLU(inplace=True)
    self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

    #Four Stages
    self.layer1 = self._make_stage(64, layers[0], stride=1)
    self.layer2 = self._make_stage(128, layers[1], stride=2)
    self.layer3 = self._make_stage(256, layers[2], stride=2)
    self.layer4 = self._make_stage(512, layers[3], stride=2)

    #Head
    self.avgpool = nn.AdaptiveAvgPool2d(1)
    self.fc = nn.Linear(512 * Bottleneck.expansion, num_classes)
    self._init_weights(zero_init_residual)

  def _make_stage(self, mid_channels, num_blocks, stride) -> nn.Sequential:
    blocks = [Bottleneck(self.in_channels, mid_channels, stride)]
    self.in_channels = mid_channels * Bottleneck.expansion
    for _ in range(1, num_blocks):
      blocks.append(Bottleneck(self.in_channels, mid_channels))
    return nn.Sequential(*blocks)

  def _init_weights(self, zero_init_residual):
    for m in self.modules():
      if isinstance(m, nn.Conv2d): #He Initialization
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
      elif isinstance(m, nn.BatchNorm2d): #Neutral Batchnorm Start
        nn.init.constant_(m.weight, 1)
        nn.init.constant_(m.bias, 0)
    if zero_init_residual: #Zero-Init Residual
      for m in self.modules():
        if isinstance(m, Bottleneck):
          nn.init.zeros_(m.bn3.weight)

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
    x = self.layer1(x)
    x = self.layer2(x)
    x = self.layer3(x)
    x = self.layer4(x)
    x = self.avgpool(x)
    x = torch.flatten(x, 1)
    x = self.fc(x)
    return x




In [9]:
def resnet50(num_classes: int = 1000) -> ResNet:
  return ResNet(layers=(3,4,6,3), num_classes=num_classes)

if __name__ == "__main__":
    model = resnet50()
    x = torch.randn(2, 3, 224, 224)
    y = model(x)
    n_params = sum(p.numel() for p in model.parameters())
    print("output:", tuple(y.shape))          # (2, 1000)
    print(f"params: {n_params:,}")            # 25,557,032 (matches torchvision)

output: (2, 1000)
params: 25,557,032
